In [ ]:
"""
Generate CMB "noise" realizations, normalized by the product of the (bin-0)
v and tau variances, matching the scale of the v/tau realizations already
generated by generate_realizations.py.

Each realization is fully independent (different random seed), so this
parallelizes across CPU cores with one realization per worker task.
Designed to be submitted as a batch job (see accompanying .sbatch script)
rather than run interactively.
"""
import os
import argparse
import shutil
import glob
import re
import numpy as np
import healpy as hp
import camb
import gc
from multiprocessing import Pool

DATA_DIR = "/mnt/beegfs/scoulombe"
nside = 2048
lmax = 3 * nside - 1
npix = hp.nside2npix(nside)

N_REALIZATIONS = 64  # target total for this run -- bump this each time you grow the dataset

def _noise_file_for(n_realizations):
    return os.path.join(DATA_DIR, f"realizations_cmbnoise_nside{nside}_n{n_realizations}.npy")

def find_source_n(target_n):
    """Find the largest existing noise file with n < target_n to copy forward.
    Mirrors find_source_n in generate_realizations.py -- see that docstring."""
    pattern = os.path.join(DATA_DIR, f"realizations_cmbnoise_nside{nside}_n*.npy")
    found = []
    for path in glob.glob(pattern):
        m = re.search(r"_n(\d+)\.npy$", path)
        if not m:
            continue
        n_candidate = int(m.group(1))
        if n_candidate < target_n and os.path.exists(_noise_file_for(n_candidate)):
            found.append(n_candidate)
    return max(found) if found else None

NOISE_FILE = _noise_file_for(N_REALIZATIONS)
NORM_PATH = os.path.join(DATA_DIR, f"norm_factors_nside{nside}.npz")


def load_noise_cl():
    if not os.path.exists(NORM_PATH):
        raise FileNotFoundError(
            f"{NORM_PATH} not found — run the notebook cell that saves "
            f"tau_std0/v_std0 first."
        )
    norm_data = np.load(NORM_PATH)
    tau_std0 = float(norm_data["tau_std0"])
    v_std0 = float(norm_data["v_std0"])

    pars = camb.set_params(
        H0=67.5, ombh2=0.022, omch2=0.122, mnu=0.06, omk=0,
        tau=0.06, As=2e-9, ns=0.965, halofit_version='mead', lmax=lmax,
    )
    results = camb.get_results(pars)
    powers = results.get_cmb_power_spectra(pars, CMB_unit=None, raw_cl=True)
    totCL = powers['total']
    cmb = totCL[:, 0]

    noise_cl = np.array(cmb[:lmax + 1], dtype=np.float64).copy()
    if len(noise_cl) < lmax + 1:
        noise_cl = np.pad(noise_cl, (0, lmax + 1 - len(noise_cl)))
    noise_cl[0] = 0.0

    variance = (tau_std0 ** 2) * (v_std0 ** 2)
    noise_cl = noise_cl / variance
    return noise_cl


def generate_one(args):
    """Generate one realization purely in memory and return it — no file I/O
    happens in worker processes at all, avoiding concurrent writes to the
    same output file on the network filesystem."""
    i, noise_cl = args
    np.random.seed(i)
    realization = hp.synfast(noise_cl, nside, lmax=lmax)
    return i, realization.astype(np.float32)


def main(n_workers):
    if os.path.exists(NOISE_FILE):
        print("Found existing noise realization file, skipping generation.")
        return

    source_n = find_source_n(N_REALIZATIONS)

    tmp_file = NOISE_FILE + ".tmp"
    out_array = np.lib.format.open_memmap(
        tmp_file, mode="w+", dtype=np.float32, shape=(N_REALIZATIONS, npix)
    )

    if source_n is not None:
        print(f"Found existing noise file for n={source_n} -- copying forward instead of recomputing")
        old_array = np.load(_noise_file_for(source_n), mmap_mode="r")
        out_array[:source_n] = old_array[:]
        out_array.flush()
        del old_array
        gc.collect()
        start_i = source_n
    else:
        print("No smaller existing noise file found -- generating all from scratch")
        start_i = 0

    noise_cl = load_noise_cl()
    new_indices = range(start_i, N_REALIZATIONS)
    tasks = [(i, noise_cl) for i in new_indices]

    print(f"Generating {len(tasks)} new CMB noise realizations ({start_i}..{N_REALIZATIONS - 1}) "
          f"using {n_workers} worker processes...")
    with Pool(n_workers) as pool:
        for i, realization in pool.imap_unordered(generate_one, tasks):
            out_array[i] = realization
            print(f"Realization {i} done", flush=True)
            del realization
            gc.collect()

    out_array.flush()
    del out_array
    gc.collect()

    shutil.move(tmp_file, NOISE_FILE)
    print("Saved:", NOISE_FILE)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--n_workers", type=int,
        default=int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4)),
        help="Number of parallel worker processes (defaults to SLURM_CPUS_PER_TASK, or CPU count)",
    )
    args = parser.parse_args()
    main(args.n_workers)